# Portfolio analytics

Assumes syncing is already done (run `ibkr_sync.ipynb`, `prices_sync.ipynb`, `cpi_sync.ipynb`, `hysa_sync.ipynb` first).
Covers: ledger replay, open/closed lots, lot returns, XIRR, taxes, HYSA counterfactual, CPI-adjusted value.

In [ ]:
from datetime import date

import polars as pl

from trades.brokers.ibkr import main
from trades.config import AppConfig, TaxRegime
from trades.ledger import counterfactuals, metrics, replay, taxes
from trades.market_data import cpi as cpi_module
from trades.market_data import hysa_rates as hysa_module
from trades.market_data import prices

AS_OF_DATE = date.today()
config = AppConfig()

## 1. Load ledger and replay

`replay_ledger` walks every ledger event in order and produces open lots, closed lots, and a cash balance.

In [ ]:
ledger = main.load_ledger(config)
result = replay.replay_ledger(ledger, config)

print(f"Open lots:     {len(result.open_lots)}")
print(f"Closed lots:   {len(result.closed_lots)}")
print(f"Cash balance:  ${result.cash_balance:,.2f}")

In [ ]:
result.open_lots

In [ ]:
result.closed_lots

## 2. Price lookup helper

Loads each symbol's price cache from disk (no network call) and builds a `price_lookup` function used throughout.

In [ ]:
symbols = result.open_lots["symbol"].unique().to_list()
price_histories = {s: prices.load_price_cache(s, config) for s in symbols}


def price_lookup(symbol: str, as_of: date) -> float | None:
    hist = price_histories.get(symbol)
    if hist is None:
        return None
    return prices.price_as_of(hist, as_of)


for symbol, hist in price_histories.items():
    print(f"{symbol}: {len(hist)} rows, latest close {hist['close'][-1]:.2f}")

In [ ]:
lot_returns_df = metrics.lot_returns(result.open_lots, price_lookup, AS_OF_DATE, config)
lot_returns_df.select(["symbol", "shares", "cost_per_share", "current_price", "days_held", "raw_return_pct", "annualized_return_pct"])

## 4. Realized gains and per-symbol metrics

In [ ]:
realized_total = metrics.realized_gain_total(result.closed_lots)
print(f"Total realized gain: ${realized_total:,.2f}")

In [ ]:
from dataclasses import asdict

all_symbols = sorted({*result.open_lots["symbol"].to_list(), *result.closed_lots["symbol"].to_list()})
sym_metrics = pl.DataFrame([
    asdict(metrics.symbol_metrics(ledger, result, s, price_lookup, AS_OF_DATE, config))
    for s in all_symbols
])
sym_metrics

## 5. Portfolio XIRR

In [ ]:
cashflows = replay.external_cashflows(ledger)
portfolio_val = replay.portfolio_value(result, price_lookup, AS_OF_DATE)

cf_df = cashflows.collect() if isinstance(cashflows, pl.LazyFrame) else cashflows
cf_dates = cf_df["event_datetime"].dt.date().to_list() + [AS_OF_DATE]
cf_amounts = cf_df["amount"].to_list() + [portfolio_val]

xirr_val = metrics.xirr(cf_dates, cf_amounts, config)
print(f"Portfolio value: ${portfolio_val:,.2f}")
print(f"Portfolio XIRR:  {xirr_val:.2%}")

## 6. Tax report

Realized gains and dividend income by year, split by tax regime.

In [ ]:
# "RESIDENT" is the default (fully taxed baseline).
# Change to "NRA" and set status_change_date if applicable.
current_regime: TaxRegime = "RESIDENT"
status_change_date = None

tax_report = taxes.annual_tax_report(
    result.closed_lots,
    ledger,
    config,
    current_regime=current_regime,
    status_change_date=status_change_date,
)
tax_report

## 7. HYSA counterfactual

"What would my balance be today if every deposit had gone into a HYSA instead?"

In [ ]:
hysa_history = hysa_module.load_hysa_rates_cache(config)
hysa_module.list_banks(hysa_history)

In [ ]:
BENCHMARK_BANK = "marcus"  # change to any bank_id from list above


def hysa_rate_lookup(as_of: date) -> float:
    rate = hysa_module.rate_as_of(hysa_history, BENCHMARK_BANK, as_of)
    return rate / 100 if rate is not None else 0.04  # fallback to 4%


hysa_series = counterfactuals.hysa_counterfactual_series(
    cashflows,
    end=AS_OF_DATE,
    rate_lookup=hysa_rate_lookup,
    days_per_year=365,
)
hysa_value = hysa_series["value"][-1]
print(f"HYSA counterfactual value: ${hysa_value:,.2f}")
print(f"Actual portfolio value:    ${portfolio_val:,.2f}")
print(f"Alpha vs HYSA:             ${portfolio_val - hysa_value:+,.2f}")

## 8. CPI-adjusted portfolio value

How much of the portfolio's nominal gain is real vs. inflation.

In [ ]:
cpi_history = cpi_module.load_cpi_cache(config)

first_event_date = ledger["event_datetime"].min().date()
cpi_then = cpi_module.cpi_as_of(cpi_history, first_event_date)
cpi_now = cpi_module.cpi_as_of(cpi_history, AS_OF_DATE)

if cpi_then and cpi_now:
    inflation = (cpi_now / cpi_then - 1) * 100
    real_portfolio_val = portfolio_val / (cpi_now / cpi_then)
    print(f"CPI {first_event_date} → {AS_OF_DATE}: {inflation:.1f}% inflation")
    print(f"Nominal portfolio value: ${portfolio_val:,.2f}")
    print(f"Real portfolio value (in {first_event_date} dollars): ${real_portfolio_val:,.2f}")